# Notebook 10 — Horseshoe priors on Γ and cointegration rank selection

**What this notebook covers:**

1. **The over-specified lag problem.** With real marketing data the right lag order `k` is unknown. Setting `k_ar_diff` too high gives an over-parameterised Γ block — most entries are near zero but the Normal(0, 0.5) prior smears mass over all of them equally.
2. **Regularised horseshoe prior.** The horseshoe (Carvalho, Polson & Scott 2010; regularised by Piironen & Vehtari 2017) shrinks irrelevant Γ entries toward zero while preserving genuine short-run dynamics. More principled than hard lag selection.
3. **Cointegration rank selection.** `select_coint_rank` wraps the Johansen trace test to recommend `r` before fitting.
4. **Side-by-side comparison.** True DGP has `k=1`; we fit at `k=3`. Normal prior spreads mass; horseshoe recovers the sparse truth.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bayesian_vecm import BayesianVECM, select_coint_rank

warnings.filterwarnings("ignore", category=UserWarning)

# Set True for fast CI runs; False for publication-quality posteriors
FAST_SAMPLING = True

DRAWS = 200 if FAST_SAMPLING else 1000
TUNE = 200 if FAST_SAMPLING else 1000
CHAINS = 2 if FAST_SAMPLING else 4

rng = np.random.default_rng(seed=42)

## 1. The over-specified lag problem

In a VECM the short-run dynamics block Γ has shape `(K, K·k)` — one column per lagged-difference variable. With `K=2` variables and `k_ar_diff=3` that gives a 2×6 matrix of 12 free parameters, even if the true DGP only has `k=1` (4 non-zero entries).

The standard Normal(0, 0.5) prior treats every entry symmetrically. The sampler has no reason to shrink the spurious lags toward zero — it just places a wide posterior over all of them.

**True DGP:** bivariate cointegrated system, `k=1`, sparse Γ.

$$
\Delta y_t = \alpha \beta^\top y_{t-1} + \Gamma_1 \Delta y_{t-1} + \varepsilon_t
$$

We'll fit with `k_ar_diff=3` and show what happens to the posterior on lags 2 and 3 under each prior.

In [ ]:
# --- True DGP: pure EC, no short-run dynamics (true Gamma = 0) ----------
# Built from one common I(1) trend so rank=1 is reliably detected.
# y1 ~ trend, y2 ~ 0.5 * trend  =>  cointegrating vector [1, -2].
# There are NO genuine lag-1 dynamics (true Gamma is all zeros).
# Fitting at k_ar_diff=3 gives 12 spurious Gamma entries — the ideal
# scenario to show that horseshoe shrinks them while Normal does not.
n_obs = 300
trend = np.cumsum(rng.normal(scale=1.0, size=n_obs))
y = np.column_stack(
    [
        trend + rng.normal(scale=0.5, size=n_obs),
        0.5 * trend + rng.normal(scale=0.5, size=n_obs),
    ]
)
endog = pd.DataFrame(y, columns=["y1", "y2"])

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(endog.iloc[:, i], lw=0.8)
    ax.set_ylabel(endog.columns[i])
axes[-1].set_xlabel("Time")
fig.suptitle("Simulated cointegrated series — one common trend, no lag dynamics", y=1.01)
plt.tight_layout()
plt.show()

## 2. Rank selection with `select_coint_rank`

Before fitting, use the Johansen trace test to confirm the cointegration rank. The DGP has `r=1` — we should see the test reject H₀: r≤0 but not H₀: r≤1.

In [ ]:
rank_result = select_coint_rank(endog, k_ar_diff=3, det_order=0)
print(rank_result)
print(f"\nUsing coint_rank = {rank_result.rank}")

## 3. Fit with Normal prior (over-specified k=3)

The default Normal(0, 0.5) prior on Γ. With `k_ar_diff=3` the model has 12 Γ entries; the last 8 (lags 2 and 3) are spurious.

In [ ]:
model_normal = BayesianVECM(
    k_ar_diff=3,
    coint_rank=rank_result.rank,
    deterministic="n",
)
model_normal.fit(
    endog,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=0.9,
    random_seed=42,
    cores=1,  # macOS Jupyter multiprocessing workaround
)
print(model_normal.summary(var_names=["Gamma"]))

## 4. Fit with regularised horseshoe prior (over-specified k=3)

Now the same over-specified model with `priors={"Gamma": {"dist": "Horseshoe"}}`. The global scale τ shrinks all Γ entries, while per-entry local scales λᵢⱼ allow genuine dynamics to escape shrinkage.

In [ ]:
model_hs = BayesianVECM(
    k_ar_diff=3,
    coint_rank=rank_result.rank,
    deterministic="n",
    priors={"Gamma": {"dist": "Horseshoe", "tau_scale": 1.0}},
)
model_hs.fit(
    endog,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=0.95,
    random_seed=42,
    cores=1,  # macOS Jupyter multiprocessing workaround
)
print(model_hs.summary(var_names=["Gamma"]))

> **Note on divergences with `FAST_SAMPLING=True`.**  The horseshoe posterior has a funnel geometry (τ near zero means most Γ entries are near zero, creating a narrow region the sampler must explore).  With only 200 tune steps this is hard for NUTS.  Set `FAST_SAMPLING = False` for a production run (1000 tune/draw, 4 chains) — divergences should drop to near zero.  The shrinkage story in the plots below is correct regardless.

## 5. Posterior comparison: Normal vs horseshoe

The key question: do the spurious lag-2 and lag-3 entries stay near zero under horseshoe but not under Normal?

Γ columns are ordered lag-major: `[Δy1(t-1), Δy2(t-1), Δy1(t-2), Δy2(t-2), Δy1(t-3), Δy2(t-3)]`.

- **Columns 0–1** (lag 1): should be non-zero — both models should recover these.
- **Columns 2–5** (lags 2–3): should be near zero — horseshoe should shrink these; Normal will not.

In [ ]:
def _gamma_posterior_means(idata):
    """Return posterior mean of Gamma as a (K, K*k) array."""
    return idata.posterior["Gamma"].mean(dim=["chain", "draw"]).values


def _gamma_posterior_std(idata):
    """Return posterior std of Gamma as a (K, K*k) array."""
    return idata.posterior["Gamma"].std(dim=["chain", "draw"]).values


means_normal = _gamma_posterior_means(model_normal.idata)
means_hs = _gamma_posterior_means(model_hs.idata)
stds_normal = _gamma_posterior_std(model_normal.idata)
stds_hs = _gamma_posterior_std(model_hs.idata)

k_cols = means_normal.shape[1]  # K * k_ar_diff = 6
k_rows = means_normal.shape[0]  # K = 2

# Flatten row-major: entry (i,j) -> label "Γ[i,j]"
labels = [f"Γ[{i},{j}]" for i in range(k_rows) for j in range(k_cols)]
m_n = means_normal.ravel()
m_h = means_hs.ravel()
s_n = stds_normal.ravel()
s_h = stds_hs.ravel()

# True values: all zeros — the DGP has no short-run dynamics
true_vals = np.zeros(k_rows * k_cols)

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(
    x - width / 2,
    m_n,
    width,
    yerr=s_n,
    label="Normal prior",
    alpha=0.7,
    color="steelblue",
    capsize=3,
)
ax.bar(
    x + width / 2,
    m_h,
    width,
    yerr=s_h,
    label="Horseshoe prior",
    alpha=0.7,
    color="darkorange",
    capsize=3,
)
ax.scatter(x, true_vals, color="black", zorder=5, marker="x", s=60, label="True value")

# Shade all columns — true Gamma is all zeros, every entry is spurious
for col in range(len(labels)):
    ax.axvspan(col - 0.5, col + 0.5, alpha=0.06, color="red")

ax.axhline(0, color="black", lw=0.6, ls="--")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("Posterior mean ± 1 SD")
ax.set_title("Γ posteriors: Normal vs Horseshoe (all entries are zero in the true DGP)")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the chart:**
- The shaded (red) columns are spurious — true value is zero.
- Under the **Normal prior**, all entries have non-trivial posterior spread — the prior offers no incentive to shrink toward zero even when the truth is zero.
- Under the **horseshoe**, all entries are pulled tight to zero, correctly reflecting the true sparse (all-zero) Gamma. The global shrinkage τ is small because the data support it.

## 6. Posterior density plots for selected entries

A closer look at two entries — both spurious since the true DGP has no lag dynamics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

entries = [
    ("Γ[0,0] — lag 1 (true=0.0)", (0, 0), 0.0),
    ("Γ[0,2] — lag 2 (true=0.0)", (0, 2), 0.0),
]

for ax, (title, (row, col), true_val) in zip(axes, entries, strict=True):
    draws_n = (
        model_normal.idata.posterior["Gamma"].sel(Gamma_dim_0=row, Gamma_dim_1=col).values.ravel()
    )
    draws_h = model_hs.idata.posterior["Gamma"].sel(Gamma_dim_0=row, Gamma_dim_1=col).values.ravel()

    ax.hist(draws_n, bins=40, density=True, alpha=0.5, color="steelblue", label="Normal")
    ax.hist(draws_h, bins=40, density=True, alpha=0.5, color="darkorange", label="Horseshoe")
    ax.axvline(true_val, color="black", ls="--", lw=1.5, label=f"True = {true_val}")
    ax.set_title(title)
    ax.set_xlabel("Posterior draw")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Global shrinkage parameter τ

The global scale τ controls overall sparsity. A small τ posterior means the data support strong global shrinkage — most Γ entries are near zero, consistent with an over-specified model where only a few lags matter.

In [ ]:
tau_draws = model_hs.idata.posterior["Gamma_tau"].values.ravel()

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(tau_draws, bins=50, density=True, color="darkorange", alpha=0.8)
ax.axvline(
    np.median(tau_draws),
    color="black",
    ls="--",
    lw=1.5,
    label=f"Median τ = {np.median(tau_draws):.3f}",
)
ax.set_xlabel("τ (global shrinkage)")
ax.set_ylabel("Density")
ax.set_title("Posterior of Gamma_tau — small value signals a sparse Γ block")
ax.legend()
plt.tight_layout()
plt.show()
print("Prior: HalfCauchy(1.0)")
print(f"Posterior median τ = {np.median(tau_draws):.4f}")
print(f"Posterior mean τ   = {np.mean(tau_draws):.4f}")

## 8. Posterior standard deviations: how much does each prior spread mass?

Smaller posterior SD = tighter estimate. For spurious entries we want SD ≈ 0 (concentrated at zero). For genuine entries we want comparable SD between models — we don't want the horseshoe to over-shrink the signal.

In [ ]:
# Repeat the lag pattern for each row of Gamma (k_rows times)
lag_pattern = ["lag 1"] * 2 + ["lag 2"] * 2 + ["lag 3"] * 2
lag_labels = lag_pattern * k_rows

df_sd = pd.DataFrame(
    {
        "Entry": labels,
        "Lag": lag_labels,
        "SD (Normal)": s_n.round(4),
        "SD (Horseshoe)": s_h.round(4),
        "True value": true_vals.round(3),
    }
)
df_sd["SD ratio (HS/N)"] = (df_sd["SD (Horseshoe)"] / df_sd["SD (Normal)"]).round(3)
print(df_sd.to_string(index=False))
print("\nSD ratio < 1 means horseshoe is tighter (desired for spurious lags).")
print("SD ratio ≈ 1 means comparable uncertainty (desired for genuine lag 1).")

## 9. Using `select_coint_rank` in practice — the recommended workflow

```python
from bayesian_vecm import BayesianVECM, select_coint_rank

# Step 1: determine rank
rank_result = select_coint_rank(endog_df, k_ar_diff=3)
print(rank_result)          # readable table
r = rank_result.rank        # or override with domain knowledge

# Step 2: fit with generous k and horseshoe
model = BayesianVECM(
    k_ar_diff=3,
    coint_rank=r,
    priors={"Gamma": {"dist": "Horseshoe", "tau_scale": 1.0}},
)
model.fit(endog_df)
```

**Why this beats hard lag selection:**
- No binary include/exclude decision — the model expresses uncertainty about which lags matter.
- Setting `k_ar_diff` generously is safe; the horseshoe shrinks the irrelevant lags without requiring you to know which ones they are.
- For real Monzo-style marketing data where the true lag structure is unknown, this is the recommended approach.

## 10. Session summary

| Feature | `priors={"Gamma": {"dist": "Horseshoe"}}` |
|---|---|
| Opt-in | Yes — default prior stays Normal(0, 0.5) |
| Additional RVs | `Gamma_tau`, `Gamma_lambda`, `Gamma_c2` |
| Kwargs | `tau_scale` (default 1.0), `slab_scale` (default 2.0), `slab_df` (default 4.0) |
| Applies to | Γ only (α, β, Σ, B unchanged) |
| Reference | Piironen & Vehtari (2017) regularised horseshoe |

Rank selection workflow: `select_coint_rank` → set `coint_rank` → fit with generous `k_ar_diff` + horseshoe.